# Coherent contaminants and the EACF veto

This tutorial compares a stochastic p-mode comb with a coherent periodic contaminant under the same observing window. A coherent signal typically occupies fewer Fourier bins, even when the window creates aliases.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from urdr import (
    CoherentSignalConfig,
    SimulationConfig,
    TimeSeries,
    add_coherent_signal,
    benchmark_coherent_veto,
    coherence_diagnostics,
    simulate_time_series,
)

## Build an exact observing window

Missing cadences stay on the uniform grid. This lets the same window function act on the oscillator, contaminant, and null simulations.

In [ ]:
size = 4096
time_days = np.arange(size) * (120.0 / 86400.0)
observed = np.ones(size, dtype=bool)
observed[900:1020] = False
template = TimeSeries(time_days, np.zeros(size), observed)

simulation = SimulationConfig(
    white_noise_sigma=0.15,
    granulation_amplitude=0.25,
    granulation_timescale_days=0.2,
    numax_uhz=1000.0,
    delta_nu_uhz=100.0,
    envelope_width_uhz=400.0,
    oscillation_amplitude=1.0,
)
template.window.diagnostics(simulation.delta_nu_uhz)

## Compare stochastic and coherent signals

The contaminant is added to the same noise-plus-granulation realisation as the null. The panels make the distinction visible in both time and frequency: the stochastic comb spreads power over many modes, whereas a coherent source concentrates it.

In [ ]:
rng = np.random.default_rng(12)
oscillator = simulate_time_series(template.window, simulation, rng)
noise = simulate_time_series(
    template.window,
    simulation,
    np.random.default_rng(12),
    include_oscillations=False,
)
coherent = add_coherent_signal(
    noise,
    CoherentSignalConfig(1000.0, 1.0, harmonics=1),
    np.random.default_rng(13),
)

def fft_power(series):
    values = np.zeros(series.time.size)
    centred = series.flux[series.observed] - np.median(series.flux[series.observed])
    values[series.observed] = centred
    frequency = np.fft.rfftfreq(values.size, d=series.cadence_seconds) * 1e6
    return frequency, np.abs(np.fft.rfft(values)) ** 2

fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex="col")
for row, (label, series, color) in enumerate([
    ("Stochastic oscillator", oscillator, "tab:blue"),
    ("Coherent line", coherent, "tab:orange"),
]):
    axes[row, 0].plot(series.time[series.observed], series.flux[series.observed], lw=0.6, color=color)
    frequency, power = fft_power(series)
    axes[row, 1].plot(frequency, power, lw=0.7, color=color)
    axes[row, 0].set_ylabel(label + "\nFlux")
    axes[row, 1].set_yscale("log")
axes[0, 0].set_title("Time series")
axes[0, 1].set_title("Zero-filled FFT power (diagnostic only)")
axes[1, 0].set_xlabel("Time [days]")
axes[1, 1].set(xlim=(500, 1500), xlabel="Frequency [µHz]", ylabel="Power")
axes[0, 1].set_ylabel("Power")
plt.tight_layout()
plt.show()

diagnostics = {
    "oscillator": coherence_diagnostics(oscillator, 1000.0, 500.0),
    "coherent": coherence_diagnostics(coherent, 1000.0, 500.0),
}
diagnostics

A coherent source should usually have a larger maximum_bin_fraction, fewer effective_bins, and lower spectral_entropy. These are diagnostics rather than universal cuts: their distributions depend on the observing window and target noise.

## Calibrate the veto for this target

The EACF threshold comes from clean null simulations. A separate concentration threshold retains a requested fraction of detected oscillation injections. The small count below keeps the tutorial quick; production calibration should normally use at least 128 realisations.

In [ ]:
metrics = benchmark_coherent_veto(
    window=template.window,
    simulation=simulation,
    contaminants={
        "single_line": CoherentSignalConfig(1000.0, 1.0),
        "three_harmonics": CoherentSignalConfig(333.3, 1.0, harmonics=3),
    },
    centre_frequencies_uhz=np.array([900.0, 1000.0, 1100.0]),
    filter_width_uhz=500.0,
    delta_nu_grid_uhz=np.linspace(90.0, 110.0, 9),
    realizations=8,
    target_false_positive_rate=0.25,
    target_signal_retention=0.9,
    max_lag_seconds=15_000.0,
    seed=42,
)
metrics